In [0]:
-- Peek at the raw orders file before loading anything
SELECT * FROM read_files(
  '/Volumes/shopstream/core/raw/orders_2026_h1.csv',
  format => 'csv',
  header => true
)
LIMIT 10;

order_line_id,order_id,customer_id,product_id,quantity,unit_price,order_ts,status,coupon_code,_rescued_data
L0000001,O000001,C00628,P0123,1,106.28,2026-02-20T07:14:43.000Z,completed,FESTIVE20,null
L0000002,O000001,C00628,P0096,1,267.14,2026-02-20T07:14:43.000Z,completed,WELCOME15,null
L0000003,O000002,C00141,P0057,2,22.09,2026-02-07T07:02:26.000Z,completed,null,null
L0000004,O000002,C00141,P0017,1,53.8,2026-02-07T07:02:26.000Z,completed,FESTIVE20,null
L0000005,O000002,C00141,P0126,1,107.67,2026-02-07T07:02:26.000Z,completed,FESTIVE20,null
L0000006,O000002,C00141,P0110,1,27.62,2026-02-07T07:02:26.000Z,completed,null,null
L0000007,O000003,C00845,P0165,2,63.33,2026-04-24T10:25:39.000Z,cancelled,null,null
L0000008,O000004,C00463,P0011,1,64.02,2026-04-27T01:56:15.000Z,returned,null,null
L0000009,O000004,C00463,P0113,1,319.88,2026-04-27T01:56:15.000Z,returned,null,null
L0000010,O000004,C00463,P0074,1,156.41,2026-04-27T01:56:15.000Z,returned,WELCOME15,null


In [0]:
-- Bronze orders: create once, then load with COPY INTO (idempotent)
CREATE TABLE IF NOT EXISTS shopstream.core.bronze_orders;

COPY INTO shopstream.core.bronze_orders
FROM '/Volumes/shopstream/core/raw/orders_2026_h1.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true', 'mergeSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
13717,13717,0


In [0]:
-- Same pattern for the two dimension files
CREATE TABLE IF NOT EXISTS shopstream.core.bronze_customers;

COPY INTO shopstream.core.bronze_customers
FROM '/Volumes/shopstream/core/raw/customers.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true', 'mergeSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');


num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
1000,1000,0


In [0]:
CREATE TABLE IF NOT EXISTS shopstream.core.bronze_products;

COPY INTO shopstream.core.bronze_products
FROM '/Volumes/shopstream/core/raw/products.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true', 'mergeSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
197,197,0


In [0]:
-- The bronze layer is in. Three tables, raw shapes preserved.
SELECT 'bronze_orders' AS table_name, COUNT(*) AS row_count FROM shopstream.core.bronze_orders
UNION ALL
SELECT 'bronze_customers', COUNT(*) FROM shopstream.core.bronze_customers
UNION ALL
SELECT 'bronze_products', COUNT(*) FROM shopstream.core.bronze_products

table_name,row_count
bronze_orders,13717
bronze_customers,1000
bronze_products,197
